# 01. Engineering Data Reality Check
**Engineering Data Processing with Python** | one-day course | approx. 35 minutes

> **EXERCISE NOTEBOOK.** Work top to bottom. Run every cell in order (Shift+Enter). Where you see `____` replace it with your answer. Where you see `# TODO`, write the code. If an error mentions `____` (for example `NameError: name '____' is not defined`), you have a blank left to fill. Blanks are four underscores.


In [ ]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

## Why this module exists

Most engineering data problems are boring and predictable. Someone typed `???` in a cell. An export changed the date
format. A sensor wrote `-999` when it lost signal. A column name has a trailing space you cannot see.
None of these are hard to fix once you have seen them. The trouble is not seeing them, then reporting a number
that is confidently wrong.

In this module you will:

1. Look at the failure modes you will meet all day.
2. Agree some rules that make your work repeatable.
3. Profile two real datasets and find what is wrong with them.
4. Write a reusable `profile()` function.
5. Decide what "clean enough" means for a specific question.

## 1. Common failure modes in engineering exports

| Failure mode | What it looks like | Why it bites |
|---|---|---|
| Numbers stored as text | `"15 kW"`, `"€1,250.00"`, `"???"` | Column will not average or sum |
| Date order | `01/02/2026`: 1 Feb or 2 Jan? | Silent. Monthly reports shift |
| Mixed units | °F from one logger, °C from another | Silent. Plausible but wrong |
| Sentinel values | `-999`, `9999`, `0` for "no reading" | Drags averages, triggers false alarms |
| Hidden whitespace and case | `"CNC-001 "`, `"cnc-001"` | Joins and counts miss records |
| Header junk | Title rows above the real header in Excel | Column names become `Unnamed: 1` |
| Duplicates | Same record exported twice, or edited and re-exported | Counts and costs inflated |
| Free-text categories | `"Corrective"`, `"CM"`, `"breakdown"` | Group totals split three ways |
| Excel "helping" | `1-3` becomes a date, leading zeros dropped | IDs no longer match |

## 2. Rules for repeatable work

1. **Raw files are read-only.** Never fix the source file by hand. Fix it in code so the fix is recorded and repeatable.
2. **Every step is in the notebook.** If you clicked something in Excel to make it work, you cannot rerun it next month.
3. **Run top to bottom.** Before you trust a result: *Runtime > Restart session and run all*. If it breaks, your
   notebook depended on something you ran out of order.
4. **Outputs go to a separate folder** (`outputs/`), never over the input.
5. **Count before and after** every cleaning step. Rows that disappear without anyone noticing are the commonest bug.

## 3. Profiling: first look at a dataset

The 3D printing dataset from the primer. Remember that `roughness` would not average.

### Exercise 1.1 [Level 1: fill the blanks]
Load the file and look at its size, its first rows and the type of each column.

In [ ]:
printer = pd.____("data/3dprinter.csv")
print("Shape (rows, columns):", printer.____)
printer.____()

In [ ]:
printer.info()   # column names, non-null counts and dtypes in one view

`roughness` shows as `object`. In pandas, `object` usually means **text**. A numeric column becomes text if even one
value cannot be read as a number. Let's find the culprit.

In [ ]:
# to_numeric with errors="coerce" turns anything that is not a number into NaN (missing).
roughness_num = pd.to_numeric(printer["roughness"], errors="coerce")

# Rows where the original had a value but the number version is missing = the bad values
bad_rows = printer[roughness_num.isna() & printer["roughness"].notna()]
bad_rows

One test row has `???` for roughness and blanks for tensile strength and elongation. Someone was clearly unsure.
Two more things to notice:

* `tension_strenght` is misspelt. If you write `printer["tension_strength"]` you get a `KeyError`.
  Misspelt column names in exports are common and they break scripts in confusing ways.
* The bad row is the last row. Exports often have junk at the end: totals, notes, partial records.

### Exercise 1.2 [Level 1: fill the blanks]
Count missing values per column, fix the column name, and convert roughness to numbers.

In [ ]:
print(printer.____().sum())

printer = printer.rename(columns={"tension_strenght": "____"})
printer["roughness"] = pd.to_numeric(printer["roughness"], errors=____)
printer.dtypes

In [ ]:
check("tension_strength" in printer.columns and printer["roughness"].dtype == "float64",
      "Column renamed and roughness is numeric. The ??? is now NaN, which pandas ignores in averages.",
      "Check the new column name spelling and errors='coerce'.")

In [ ]:
# Now this works
printer.groupby("material")["roughness"].mean()

**Decision point.** We turned `???` into missing. We did not delete the row, and we did not guess a value.
That is usually the safest default: keep the evidence, make the gap visible, decide later what to do with it.

## 4. A bigger dataset: concrete compressive strength

### Exercise 1.3 [Level 1: fill the blanks]

In [ ]:
concrete = pd.read_csv("data/____")
print(concrete.shape)
concrete.____().T     # .T flips the table so each column is a row. Easier to read.

The numbers look sensible (cement 102 to 540 kg/m³, strength 2 to 83 MPa). The column names are painful.
Print them with `repr` so hidden spaces show up.

In [ ]:
for name in concrete.columns:
    print(repr(name))

Look closely: `'Water  (component 4)...'` has **two** spaces and the strength column ends with a **space** before the
closing quote. Type the name as you see it in Excel and it will not match.

### Exercise 1.4 [Level 1: fill the blanks]
Replace the column names with short, consistent names that include units. Two are left for you.

In [ ]:
new_names = [
    "cement_kg_m3",
    "slag_kg_m3",
    "fly_ash_kg_m3",
    "____",
    "superplasticiser_kg_m3",
    "coarse_agg_kg_m3",
    "fine_agg_kg_m3",
    "age_days",
    "____",
]
print(len(new_names), "new names for", len(concrete.columns), "columns")
concrete.columns = new_names
concrete.head()

In [ ]:
check(list(concrete.columns)[3] == "water_kg_m3" and list(concrete.columns)[-1] == "strength_mpa",
      "Readable names. Water/cement ratio is now easy to calculate.",
      "Expected 'water_kg_m3' and 'strength_mpa'.")

Assigning a list of names relies on the **order** of columns. It is fine here because we checked the order by eye.
In a script that runs on new exports every month, renaming with a dictionary (`old name: new name`) is safer,
because if a column moves, the dictionary still finds it. You will do that in Module 03.

### Exercise 1.5 [Level 1: fill the blanks]
Two quick engineering checks: are there duplicate rows, and what is the water/cement ratio range?

In [ ]:
print("Duplicate rows:", concrete.____().sum())

concrete["w_c_ratio"] = concrete["____"] / concrete["____"]
concrete["w_c_ratio"].describe()

25 duplicate rows. Are they errors, or genuine repeat tests of the same mix at the same age? You cannot tell from the
data alone. Keep that question for Module 04.

## 5. A reusable profiling function

You will profile every new dataset. Write it once.

### Exercise 1.6 [Level 2: finish the code]

Complete `profile(df)` so it returns one row per column of `df` with:
`dtype`, `missing` (count), `missing_pct`, `unique` (number of distinct values) and `example` (first non-missing value).

Useful pieces: `df.dtypes`, `df.isna().sum()`, `df.isna().mean()`, `df.nunique()`. The `example` line is written for you.

In [ ]:
def profile(df):
    """One row per column: type, missing count and %, distinct values, an example value."""
    summary = pd.DataFrame({
        # TODO: "dtype": ...  (hint: df.dtypes.astype(str))
        # TODO: "missing": ...
        # TODO: "missing_pct": ...  (percentage, rounded to 1 decimal place)
        # TODO: "unique": ...
        "example": df.apply(lambda col: col.dropna().iloc[0] if col.notna().any() else None),
    })
    return summary

profile(printer)

In [ ]:
p = profile(printer)
check(set(["dtype", "missing", "missing_pct", "unique", "example"]).issubset(p.columns) and p.loc["roughness", "missing"] == 1,
      "profile() works. Keep it, you will use it again.",
      "Need columns dtype, missing, missing_pct, unique, example. roughness should show 1 missing.")

## 6. A preview of the main dataset for today

For the rest of the day we work with data from **Kilbrack Engineering Ltd**, a fictional manufacturer with plants in
Cork, Galway and Limerick. The data is synthetic, but the faults planted in it are the common ones in real maintenance and asset data.
You will meet:

* `work_orders_cmms.csv`: maintenance work orders from the current maintenance system (CMMS)
* `work_orders_legacy.xlsx`: older work orders from the spreadsheet it replaced
* `asset_register.xlsx`: the list of assets
* `sensor_readings.csv`: condition monitoring data from compressors and pumps

Run your profiler on the work orders.

In [ ]:
wo = pd.read_csv("data/work_orders_cmms.csv")
print(wo.shape)
profile(wo)

In [ ]:
# How many different spellings of work order type?
wo["wo_type"].value_counts()

Every column is `object` (text), including hours, cost and dates. There are 12 spellings of 3 work order types.
And `unique` for `wo_id` is lower than the number of rows, so some work orders appear more than once.
That is the next three modules sorted.

## 7. What is "clean enough"?

There is no such thing as perfectly clean data. There is data that is clean enough **for a specific question**.
Cleaning everything to perfection wastes days. Cleaning nothing produces nonsense.

### Exercise 1.7 [Level 3: think it through, then compare with Gemini]

The operations manager asks: **"Which asset type costs us the most in corrective maintenance?"**

In the markdown cell below (double-click to edit), list:
1. Which columns must be right to answer this?
2. Which problems in those columns would change the answer?
3. Which columns can you ignore for this question?

Then ask Gemini the same thing. Paste in the output of `profile(wo)` and `wo.head()`, not the whole file.
Compare. What did it spot that you missed? Did it assume anything that is not true (for example a column
that does not exist)?

*Your answer here (double-click to edit):*

1. Columns that must be right:
2. Problems that would change the answer:
3. Columns I can ignore:
4. What Gemini added / got wrong:

## Summary

* `object` dtype on a numeric column means at least one value is text. `pd.to_numeric(..., errors="coerce")` finds it.
* Print column names with `repr()` to see hidden spaces.
* Duplicates are a judgement call, not an automatic delete.
* Profile every new dataset with the same function.
* Define "clean enough" against the question you need to answer.

Next: **02 Loading Engineering Datasets**.